# 05 — Graph-only image retrieval baseline

This notebook proves the **G** branch of the experiment can run end to end: taxonomy graph → concept vectors → pooled image vectors → the same Iconclass retrieval evaluator used by the visual baseline. The committed fixture is a smoke test, not a research result.

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from caypollard.benchmarks.iconclass_retrieval import evaluate_iconclass_retrieval
from caypollard.embeddings.store import EmbeddingTable
from caypollard.graphs.embeddings import adjacency_svd_embeddings, aggregate_concept_embeddings_to_images
from caypollard.graphs.iconclass import build_parent_index, child_edges, parse_notations
from caypollard.retrieval import neighbor_overlap_at_k

In [ ]:
graph_records = parse_notations(ROOT / "data/samples/iconclass_notations_fixture.txt")
edges = child_edges(graph_records)
parents = build_parent_index(edges)
concepts = adjacency_svd_embeddings(edges, dimension=4, seed=42)

records = [
    {"id": "a", "iconclass": ["25G41"], "split": "test"},
    {"id": "b", "iconclass": ["25G411"], "split": "test"},
    {"id": "c", "iconclass": ["25G411"], "split": "test"},
    {"id": "d", "iconclass": ["25G412"], "split": "test"},
    {"id": "e", "iconclass": ["25G41(+1)"], "split": "test"},
]
graph_images = aggregate_concept_embeddings_to_images(records, concepts, parents)
graph_images.metadata

In [ ]:
graph_summary, graph_queries = evaluate_iconclass_retrieval(
    graph_images, records, parents, query_split="test", candidate_split="test"
)
graph_summary

## Representation disagreement

Neighbour overlap is itself a useful diagnostic. A graph representation that simply reproduces the visual neighbourhood contributes little to fusion; disagreement creates the space in which the research hypothesis can actually be tested.

In [ ]:
visual_fixture = EmbeddingTable(
    ids=("a", "b", "c", "d", "e"),
    vectors=np.asarray([
        [1.0, 0.0],
        [0.0, 1.0],
        [0.0, 0.98],
        [0.1, 0.9],
        [0.8, 0.2],
    ], dtype=np.float32),
    metadata={"fixture": True},
)
mean_overlap, per_query_overlap = neighbor_overlap_at_k(visual_fixture, graph_images, k=2)
mean_overlap, per_query_overlap

## Interpretation constraint

The taxonomy-SVD graph baseline is trained from the same Iconclass hierarchy used for graded evaluation. Its score is therefore a **sanity/control result**, not independent evidence that a KG improves historical understanding. The stronger test begins when richer relations (book, creator, printer, place, date, collection) are embedded and evaluated on held-out iconographic retrieval and cross-collection transfer.

## Hubness / degree diagnostic

A graph embedding can look semantically useful merely because high-degree entities become universal neighbours. Caypollard therefore records the relationship between graph degree and top-k neighbour occurrence.

In [ ]:
from caypollard.graphs.diagnostics import degree_hubness_correlation
from caypollard.graphs.triples import read_triples_tsv
from caypollard.graphs.embeddings import rdf2vec_ppmi_embeddings

context_triples = tuple(
    t for t in read_triples_tsv(ROOT / "data/samples/context_triples_fixture.tsv")
    if t[1] != "has_iconclass"
)
context_vectors = rdf2vec_ppmi_embeddings(
    context_triples, dimension=4, walks_per_entity=8, walk_length=6, window=2, seed=42
)
degree_hubness_correlation(context_triples, context_vectors, k=2)